# ACF sur signaux régulièrement échantillonnés : Pastas vs acf_nufft

Ce notebook illustre le chemin rapide d'`acf_nufft` dédié aux signaux **régulièrement**
échantillonnés (`compute_acf_regular_fft`, `compute_acf_rectangle_fft`,
`compute_acf_gaussian_fft` -- voir `src/acf_nufft/fft_acf.py`), sur 5 types de signaux
synthétiques : sinus, sinus bruité, exponentielle décroissante, constant (cas dégénéré),
et signal carré.

Contrairement au notebook `notebook/pastas_vs_nufftact.ipynb` (qui traite le cas
**irrégulier** via NUFFT), ce notebook compare directement Pastas à la version FFT
classique du package, pour les 3 méthodes de Pastas : `"regular"` (pas de noyau),
`"rectangle"` et `"gaussian"`.

In [ ]:
# Installe acf_nufft (numpy, pandas, numba, scipy, finufft inclus) + Pastas/matplotlib
# (extra "benchmark"). Remplace l'URL par ton repo une fois poussé sur GitHub.
!pip install -q "acf-nufft[benchmark] @ git+https://github.com/TODO-username/nufftacf.git"


In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import pastas as ps

from acf_nufft import (
    compute_acf_regular_fft,
    compute_acf_rectangle_fft,
    compute_acf_gaussian_fft,
)

import acf_nufft
print(f"Version de acf_nufft: {acf_nufft.__version__}")
print(f"Version de pastas: {ps.__version__}")


## Génération des séries de test

In [ ]:
def generate_series(n_days: int, signal_type: str = "sinus", **params) -> pd.Series:
    """Génère une série régulière (delta_t=1 jour) sur l'un des 5 types de
    signaux utilisés dans ce notebook."""
    t = np.arange(n_days, dtype=float)
    if signal_type == "sinus":
        f = params.get("f", 1.0 / (n_days / 10))
        data = np.sin(2 * np.pi * f * t)
    elif signal_type == "sinus_bruit":
        f = params.get("f", 1.0 / (n_days / 10))
        rng = np.random.default_rng(params.get("seed", 0))
        data = np.sin(2 * np.pi * f * t) + rng.standard_normal(n_days) * 0.1
    elif signal_type == "exponentielle":
        tau_0 = params.get("tau_0", n_days / 5)
        data = np.exp(-t / tau_0)
    elif signal_type == "constant":
        # Cas degenere : variance nulle -> l'ACF est mathematiquement indefinie
        # (division par zero). Sert a verifier que le package echoue proprement
        # (NaN partout, sans planter) plutot que silencieusement faux.
        data = np.ones(n_days)
    elif signal_type == "carre":
        f = params.get("f", 1.0 / (n_days / 10))
        data = np.sign(np.sin(2 * np.pi * f * t))
    else:
        raise ValueError(f"signal_type {signal_type!r} inconnu")
    idx = pd.date_range("2000-01-01", periods=n_days, freq="D")
    return pd.Series(data=data, index=idx, name=f"STS_{signal_type}")


SIGNAL_TYPES = ["sinus", "sinus_bruit", "exponentielle", "constant", "carre"]
N_days_list = [3650]


## Comparaison Pastas vs acf_nufft, pour chaque signal et chaque méthode

Pour chaque signal, on calcule l'ACF avec Pastas et avec `acf_nufft` (version FFT,
régulière) pour les 3 méthodes (`regular`, `rectangle`, `gaussian`), puis on les
superpose.

In [ ]:
FFT_FUNCS = {
    "regular": compute_acf_regular_fft,
    "rectangle": compute_acf_rectangle_fft,
    "gaussian": compute_acf_gaussian_fft,
}


def compute_comparison(sts: pd.Series, lags_max: int = 365, bin_width: float = 0.5):
    """Calcule Pastas et acf_nufft (fft) pour les 3 methodes, sur une serie."""
    lags = np.arange(0.0, lags_max + 1.0)
    t = (sts.index - sts.index[0]) / pd.Timedelta(days=1)
    t = t.to_numpy(dtype=float)
    x = sts.to_numpy()

    out = {}
    for method in ["regular", "rectangle", "gaussian"]:
        t0 = time.time()
        acf_pastas = ps.stats.acf(
            sts, lags=lags, bin_method=method, bin_width=bin_width, min_obs=0
        )
        dt_pastas = time.time() - t0

        func = FFT_FUNCS[method]
        t0 = time.time()
        if method == "regular":
            c, b = func(lags, t, x)
        else:
            c, b = func(lags, t, x, bin_width=bin_width)
        dt_fft = time.time() - t0

        out[method] = dict(
            pastas_lags=acf_pastas.index.days.to_numpy(),
            pastas_c=acf_pastas.to_numpy(),
            fft_lags=lags,
            fft_c=c,
            t_pastas=dt_pastas,
            t_fft=dt_fft,
        )
    return out


def plot_comparison(out: dict, title: str):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for ax, method in zip(axes, ["regular", "rectangle", "gaussian"]):
        r = out[method]
        ax.plot(r["pastas_lags"], r["pastas_c"], "-", color="tab:blue", lw=2,
                 label=f"Pastas ({method}) -- {r['t_pastas']*1000:.1f} ms")
        ax.plot(r["fft_lags"], r["fft_c"], "--", color="tab:red", lw=2,
                 label=f"acf_nufft fft ({method}) -- {r['t_fft']*1000:.2f} ms")
        ax.set_xlabel("Lag [days]")
        ax.set_title(method)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
for N_days in N_days_list:
    for signal_type in SIGNAL_TYPES:
        sts = generate_series(N_days, signal_type)
        print(f"--- {signal_type} (N={N_days}) ---")
        if signal_type == "constant":
            # Pastas leve une erreur/avertissement sur une serie a variance nulle ;
            # on se contente ici de verifier que acf_nufft renvoie bien des NaN,
            # sans tracer de figure (rien de visuel a montrer pour un cas degenere).
            t = np.arange(N_days, dtype=float)
            x = sts.to_numpy()
            lags = np.arange(0.0, 366.0)
            for method, func in FFT_FUNCS.items():
                kwargs = {} if method == "regular" else dict(bin_width=0.5)
                with np.errstate(divide="ignore", invalid="ignore"):
                    c, b = func(lags, t, x, **kwargs)
                assert np.all(np.isnan(c)), f"{method}: attendu NaN partout (variance nulle)"
            print("  OK : ACF bien NaN partout pour le signal constant (variance nulle).")
            continue
        out = compute_comparison(sts)
        plot_comparison(out, f"{signal_type} (N={N_days})")


## Mini-benchmark (illustration rapide)

Version allégée (peu de tailles, peu de répétitions) du benchmark complet
de `benchmark/benchmark_acf_regular.py`, juste pour illustrer dans ce notebook
le gain de Pastas O(n²) vs acf_nufft O(n log n). Les figures complètes (avec fit
robuste, bootstrap, et beaucoup plus de points) sont dans `benchmark/` --
voir le README.

In [ ]:
def quick_benchmark(n_points_list, method="gaussian", bin_width=0.5, pastas_max_n=8000):
    # NB: l'index reste numerique (pas de DatetimeIndex) pour acf_nufft -- une
    # vraie plage de dates depuis 2000 deborderait la limite de pandas (~année
    # 2262) bien avant n=130000. On ne construit un DatetimeIndex (pour Pastas)
    # que pour les n suffisamment petits, sous la garde du cap ci-dessous.
    lags = np.arange(0.0, 366.0)
    rows = []
    for n in n_points_list:
        x = np.random.default_rng(0).standard_normal(n)
        t = np.arange(n, dtype=float)

        func = FFT_FUNCS[method]
        t0 = time.time()
        if method == "regular":
            func(lags, t, x)
        else:
            func(lags, t, x, bin_width=bin_width)
        rows.append(dict(n=n, algo="acf_nufft (fft)", time_s=time.time() - t0))

        if n <= pastas_max_n:
            idx = pd.date_range("2000-01-01", periods=n, freq="D")
            sts = pd.Series(x, index=idx)
            t0 = time.time()
            ps.stats.acf(sts, lags=lags, bin_method=method, bin_width=bin_width, min_obs=0)
            rows.append(dict(n=n, algo="Pastas", time_s=time.time() - t0))
    return pd.DataFrame(rows)


n_points_list = [500, 2000, 8000, 32000, 130000]
bench_df = quick_benchmark(n_points_list, method="gaussian", pastas_max_n=8000)

plt.figure(figsize=(8, 5))
for algo, g in bench_df.groupby("algo"):
    plt.plot(g["n"], g["time_s"], "o-", label=algo)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Nombre de points")
plt.ylabel("Temps [s]")
plt.title("Mini-benchmark -- gaussien, donnees regulieres")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

print(bench_df)
